# Flows spike I — flow types and the vector field

*The `explore-flows` spike, published as a walkthrough of the draft types.*

```{admonition} This is a spike, not the summer4 API
:class: warning

Everything on this page lives in `explorations/flows/` and is imported as
`explorations.flows.prototype`. It is **not** importable from `summer4`, it is
not stable, and names or signatures may change without notice. It is published
here because it is real, tested, executable design work — see
{doc}`../explorations` for the conclusions it produced and
{doc}`../../evaluation/if-promoted` for what promoting it would mean.

The code cells below are byte-identical to `explorations/flows/01-flows.ipynb`, which
`pixi run explore-flows` executes as a test. `tests/test_flows_docs_sync.py`
fails if the two drift apart.
```

A **flow** is one named object that *actualizes* to many edges against a
`PropertyMap`. Those edges compile into a **vector field** `dy/dt` — not an ODE
solver. Three flow kinds cover movement in an ODE sense:

| Type | Meaning | Default law |
| --- | --- | --- |
| `TransitionFlow` | between compartments | `rate * y[src]` onto dest |
| `ExitFlow` | out of the system | `rate * y[src]` |
| `EntryFlow` | into the system | absolute `rate` |

Rates may be a scalar, a field on a derived-parameter struct, or another flow's
output. The taxonomy layer this builds on is documented in
{doc}`../../user/03-selectors-and-queries` and
{doc}`../../user/04-ragged-stratification`.

## 1. The compartment table (`Property`, `PropertyMap`)

A `Property` is a named set of mutually exclusive traits. A `PropertyMap` is the
ragged table of compartments those properties create. Each **row** is one
compartment; `select` returns its integer indices.

We build an SIR model, fully stratified by age and a **3×3 location grid**, with
severity only on `I` (a ragged column). Location traits are `r{row}c{col}` so the
grid geometry is visible in the names.

In [ ]:
from typing import NamedTuple

import numpy as np

from explorations.flows.prototype import (
    EntryFlow,
    ExitFlow,
    FlowModel,
    TraitChain,
    TraitMatrix,
    TransitionFlow,
    actualize,
    derived_refs,
)
from summer4 import Everything, Property, PropertyMap

GRID = 3
CELLS = tuple(f"r{row}c{col}" for row in range(GRID) for col in range(GRID))
assert CELLS == ("r0c0", "r0c1", "r0c2", "r1c0", "r1c1", "r1c2", "r2c0", "r2c1", "r2c2")

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
location = Property("location", CELLS)
severity = Property("severity", ("mild", "severe"))

pm = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(location)
    .stratify(severity, where=state["I"])
)
# S×3×9 + I×3×9×2 + R×3×9 = 27 + 54 + 27 = 108
assert pm.size == 108
assert pm.select(state["S"]).size == 27
assert pm.select(state["I"]).size == 54
assert pm.select(severity.absent()).size == 54  # S and R have no severity
pm.size

`PropertyMap.select(selector)` is the only query the join needs. Selectors are
the taxonomy algebra (`state["S"]`, `age["0-4"] & location["r1c1"]`, `severity.absent()`,
`Everything()`). A flow stores those selectors; it does **not** expand into one
Python object per edge (that was the summer2 style).

## 2. `FlowModel` — a named bag of flows over one map

`FlowModel(pmap)` is a thin container. `add_flow` requires unique names and
returns a `FlowRef` used in later rates (`death.sum_over(location)`).
`compile()` actualizes every join **once**, then returns `vf(t, y, params)`.

In [ ]:
model = FlowModel(pm)
assert model.pmap is pm
assert model.flows == []

## 3. Identity join — `TransitionFlow` for infection

`TransitionFlow(name, source, dest, rate)` is the summer3 constructor order.

**Bound properties** are names mentioned in either selector. Here `state` is bound
(`S` and `I`). **Free properties** are the rest that exist on both sides — `age`
and `location`. The join pairs `s` with `d` when they agree on every free
property, so one infection flow touches all 27 `S` compartments and the matching
`I` rows.

`I` also has `severity`, which `S` does not. That is a **dest-only** axis: each
`S` fans out to mild and severe `I` of the same age and cell. Default weights are
`1/N`. Uneven ratios are a property of **this flow** (`split=`), not of a
`Stratification` (summer2's `set_population_split`).

In [ ]:
class Derived(NamedTuple):
    foi: float
    death_rate: float


D = derived_refs(Derived)

infection = TransitionFlow(
    "infection",
    state["S"],
    state["I"],
    D.foi,
    split={severity: {"mild": 0.8, "severe": 0.2}},
)
model.add_flow(infection)

inf_edges = actualize(infection, pm)
assert inf_edges.n_edges == 54  # 27 S × 2 severity dests
src0 = int(inf_edges.src_idx[0])
w0 = inf_edges.weight[inf_edges.src_idx == src0]
np.testing.assert_allclose(sorted(w0.tolist()), [0.2, 0.8])
inf_edges.n_edges

`D.foi` is a `FieldRef` — a lazy path, not a number. `derived_refs(Derived)`
builds those paths from the NamedTuple schema so tab-complete only offers
real fields. At vector-field time the compiled function calls
`compute_derived_params(params, y=y)` and walks `.foi` on the returned
`Derived`. That is how a rate can depend on the current state (force of
infection) without a special `InfectionFrequencyFlow` class.

Recovery is the same identity join, `I→R`. Severity is now **source-only**
(present on `I`, absent on `R`): both mild and severe `I` of a given age and cell
drain into the single matching `R`. That is many-to-one; no `split`.

In [ ]:
recovery = TransitionFlow("recovery", state["I"], state["R"], 0.1)
model.add_flow(recovery)
rec_edges = actualize(recovery, pm)
assert rec_edges.n_edges == 54  # each I row has one R dest
assert rec_edges.dest_idx is not None
assert rec_edges.dest_idx.size == rec_edges.src_idx.size

## 4. Ageing — one named flow, not one flow per band

Ageing looks like several transitions (`0-4→5-9`, `5-9→10+`, nothing out of `10+`).
That does **not** require several named flows. `TraitChain` is a pairing override:
it removes `age` from the free set and emits one identity-join per listed pair.
Leftover properties (state, location, severity) still match, so `I` mild in cell
`r1c1` ages into `I` mild in the same cell.

Per-band rates (`1 / width`) live on the chain as optional `rates`, stored on
each edge's `scale` — the same slot `TraitMatrix` uses for nonzero entries.
The flow `rate` (here `1.0`) still multiplies. Unequal widths would be
`rates=(1/5, 1/10)` on this same flow; you would not add `ageing_04` and
`ageing_59` as separate names.

Two identity `TransitionFlow`s (`age["0-4"]→age["5-9"]` and …) remain valid.
They are just more verbose when the pairing is a single chain.

In [ ]:
ageing = TransitionFlow(
    "ageing",
    age.present(),
    age.present(),
    1.0,
    pairing=TraitChain(
        age,
        (("0-4", "5-9"), ("5-9", "10+")),
        rates=(1.0 / 5.0, 1.0 / 5.0),
    ),
)
model.add_flow(ageing)
age_edges = actualize(ageing, pm)
labels = pm.labels()
assert age_edges.src_idx is not None
# 2 steps × 36 compartments per age band (9 loc × (S + R + 2 I))
assert age_edges.n_edges == 72
assert not any("age=10+" in labels[int(i)] for i in age_edges.src_idx)
assert set(np.round(age_edges.scale, 8).tolist()) == {0.2}
age_edges.n_edges

## 5. Migration — `TraitMatrix` on a 3×3 grid of neighbours

`TraitMatrix(location, M)` is the pairing for a (possibly sparse) transition
matrix. Convention: **`M` is dest × source**. Entry `M[j, i]` is the per-capita
rate from location trait `i` to trait `j`. Zero entries are not edges.

We want movement only between **immediate 4-neighbours** (up/down/left/right),
no diagonals, no wrap-around:

```
r0c0 — r0c1 — r0c2
  |      |      |
r1c0 — r1c1 — r1c2
  |      |      |
r2c0 — r2c1 — r2c2
```

Implementation: allocate a 9×9 zero matrix, then for each cell write `hop_rate`
into the four in-bound dests that exist. A 3×3 grid has 12 undirected adjacencies
and therefore **24 directed** location edges. `TraitMatrix` then identity-joins
each nonzero pair; leftover state/age/severity stay matched, so an infectious
mild child in `r1c1` only moves to the same compartment in `r0c1`, `r1c0`,
`r1c2`, and `r2c1`.

In [ ]:
def parse_cell(name: str) -> tuple[int, int]:
    """Decode ``r{row}c{col}`` into integer grid coordinates."""
    return int(name[1]), int(name[3])


def neighbor_matrix(cells: tuple[str, ...], hop_rate: float) -> np.ndarray:
    """Dest×source matrix: ``hop_rate`` on 4-neighbour pairs, else 0."""
    index = {name: i for i, name in enumerate(cells)}
    matrix = np.zeros((len(cells), len(cells)))
    for src in cells:
        row, col = parse_cell(src)
        for d_row, d_col in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            dest = f"r{row + d_row}c{col + d_col}"
            if dest in index:
                matrix[index[dest], index[src]] = hop_rate
    return matrix


HOP = 0.05
mig = neighbor_matrix(CELLS, HOP)
assert mig.shape == (9, 9)
assert int(np.count_nonzero(mig)) == 24
assert mig[CELLS.index("r0c1"), CELLS.index("r0c0")] == HOP  # east of NW
assert mig[CELLS.index("r0c2"), CELLS.index("r0c0")] == 0.0  # not a neighbour
assert mig[CELLS.index("r1c1"), CELLS.index("r0c0")] == 0.0  # diagonal
np.testing.assert_allclose(mig, mig.T)  # undirected grid, equal hops both ways

migration = TransitionFlow(
    "migration",
    location.present(),
    location.present(),
    1.0,
    pairing=TraitMatrix(location, mig),
)
model.add_flow(migration)
mig_edges = actualize(migration, pm)
# 24 directed location hops × 12 compartments per cell
assert mig_edges.n_edges == 24 * (pm.size // 9)
assert set(np.round(mig_edges.scale, 8).tolist()) == {HOP}
mig_edges.n_edges

## 6. Boundary flows — `ExitFlow`, `EntryFlow`, and `FlowRef`

`ExitFlow(name, source, rate)` has no destination. `Everything()` means every
row. The rate here is `D.death_rate`, resolved against the derived struct each
step.

`EntryFlow(name, dest, rate)` is always absolute. Replacement births read
**another flow's contribution**: `add_flow` returns a `FlowRef`.
`death.sum_over(location)` reduces that contribution onto the nine location
traits. At compile time flows are topologically ordered so `death` is
evaluated before `birth`. Cycles raise. The dest selector
`age["0-4"] & state["S"]` is one young-S cell per location, so dest weights
are `1.0` (not a global `1/9`).

In [ ]:
death = model.add_flow(ExitFlow("death", Everything(), D.death_rate))
model.add_flow(EntryFlow("birth", age["0-4"] & state["S"], death.sum_over(location)))
assert [flow.name for flow in model.flows] == [
    "infection",
    "recovery",
    "ageing",
    "migration",
    "death",
    "birth",
]
birth_edges = actualize(model.flows[-1], pm)
assert birth_edges.n_edges == 9
np.testing.assert_allclose(birth_edges.weight, 1.0)

## 7. Derived parameters and the vector field

`compile(derived_fn=...)` calls `derived_fn(params, y=y, t=t)` every step.
Infection FOI is frequency-dependent (`contact × I / N`) and stored on the
returned `Derived` so `D.foi` can find it. Flows that only need `params` still
go through this hook when a `FieldRef` is used; a bare float rate (recovery
`0.1`) does not.

The compiled field scatters `-mass` at sources and `+mass` at dests.
Location-dependent replacement returns every death to the young-S cell of the
same location, so `sum(dy) == 0`. A death+birth-only field checks that each
of those dests receives that location's death total (not a global `1/9`).

In [ ]:
I_idx = pm.select(state["I"])


def compute_derived_params(params, y=None, t=None):
    infected = y[I_idx].sum()
    population = y.sum()
    return Derived(
        foi=params["contact"] * infected / population,
        death_rate=params["death_rate"],
    )


y = np.zeros(pm.size)
for i, cell in enumerate(CELLS):
    y[pm.select(location[cell] & state["S"])] = 50.0 * (i + 1)
y[pm.select(state["I"] & severity["mild"])] = 4.0
y[pm.select(state["I"] & severity["severe"])] = 1.0

params = {"contact": 0.4, "death_rate": 0.01}
vf = model.compile(derived_fn=compute_derived_params)
dy = vf(0.0, y, params)

assert dy.shape == (pm.size,)
np.testing.assert_allclose(dy.sum(), 0.0, atol=1e-12)
assert I_idx.size == 54
top_s = pm.select(age["10+"] & state["S"])
assert float(dy[top_s].sum()) > 0.0  # ageing arrives; no outgoing ageing

birth_only = FlowModel(pm)
death_ref = birth_only.add_flow(ExitFlow("death", Everything(), D.death_rate))
birth_only.add_flow(EntryFlow("birth", age["0-4"] & state["S"], death_ref.sum_over(location)))
dy_rep = birth_only.compile(derived_fn=compute_derived_params)(0.0, y, params)
location_births = []
for cell in CELLS:
    expected = params["death_rate"] * float(y[pm.select(location[cell])].sum())
    dest = int(pm.select(age["0-4"] & state["S"] & location[cell])[0])
    young_death = params["death_rate"] * float(y[dest])
    np.testing.assert_allclose(dy_rep[dest], expected - young_death)
    location_births.append(expected)
assert max(location_births) > min(location_births)
np.testing.assert_allclose(dy_rep.sum(), 0.0, atol=1e-12)
dy.sum()

## 8. JAX / `PropertyData`

`backend="jax"` uses the same actualized index arrays with `jnp` scatter.
`PropertyData` is the explore-datatypes pairing: a JAX array whose last axis is
compartments, with the `PropertyMap` as digest-hashed pytree aux. jit over a
wrapped state returns a wrapped `dy`.

In [ ]:
import jax
import jax.numpy as jnp

from explorations.flows.propertydata import PropertyData

vf_jax = model.compile(derived_fn=compute_derived_params, backend="jax")
jvf = jax.jit(vf_jax)
dy_j = np.asarray(jvf(0.0, jnp.asarray(y), params))
np.testing.assert_allclose(dy_j, dy, rtol=1e-5)

wrapped = PropertyData.wrap(pm, y)
dy_pd = jvf(0.0, wrapped, params)
np.testing.assert_allclose(np.asarray(dy_pd.data), dy, rtol=1e-5)
assert dy_pd.pmap.size == pm.size

---

Continue with {doc}`02-derived-rates-and-adjustments`, which adds nested derived
parameters, the `adjust=` pipeline, and time stepping.